# GPUPerf: Universal Multi-Scale Training Engine
## 🚀 Fine-Tune Across 3B, 7B, 14B, 27B, and 32B Architectures

This turnkey notebook trains and evaluates **GPUPerf** across any model tier using **100% real physical GPU measurements** harvested across 4 distinct NVIDIA GPU generations:
- 🟦 **Turing (Tesla T4, `sm_75`)**: 40 SMs, 320 GB/s GDDR6
- 🟩 **Ampere (NVIDIA A100-SXM4-80GB, `sm_80`)**: 108 SMs, 2,039 GB/s HBM2e
- 🟧 **Ada Lovelace (NVIDIA L4, `sm_89`)**: 58 SMs, 300 GB/s GDDR6
- 🟪 **Blackwell (RTX PRO 6000 Blackwell Server Edition, `sm_120`)**: 188 SMs, 500 GB/s

### Supported Model Scales:
- **3B:** `Qwen/Qwen2.5-Coder-3B-Instruct` (Fastest, ~10 min on Colab T4/L4)
- **7B:** `Qwen/Qwen2.5-Coder-7B-Instruct` (High-reasoning, ~25 min on T4, ~12 min on A100)
- **14B:** `Qwen/Qwen2.5-Coder-14B-Instruct` (Requires L4 24GB or A100 40GB/80GB)
- **27B / 32B:** `Qwen/Qwen3.6-27B` or `Qwen/Qwen2.5-Coder-32B-Instruct` (Requires A100 40GB/80GB)

In [ ]:
# 1. Verify Attached Hardware
!nvidia-smi

In [ ]:
# 2. Fast Install Unsloth & Modern Fine-Tuning Stack
%%capture
!pip install --no-deps "xformers<0.0.29" triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. Clone Repository & Setup Dataset
import os
if not os.path.exists("GPUPerf"):;
    !git clone https://github.com/sarimahsan/GPUPerf.git
%cd GPUPerf
!git pull
!ls -lh ./data

In [ ]:
# 4. SELECT MODEL SCALE & AUTO-TUNE HYPERPARAMETERS
# Choose model scale: '3b', '7b', '14b', '27b', '32b'
MODEL_SCALE = "3b"  # @param ["3b", "7b", "14b", "27b", "32b"]

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

MODEL_MAP = {
    "3b": "Qwen/Qwen2.5-Coder-3B-Instruct",
    "7b": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "14b": "Qwen/Qwen2.5-Coder-14B-Instruct",
    "27b": "Qwen/Qwen3.6-27B",
    "32b": "Qwen/Qwen2.5-Coder-32B-Instruct",
}

model_name = MODEL_MAP[MODEL_SCALE]
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
gpu_name = torch.cuda.get_device_name(0)

print(f"Detected GPU: {gpu_name} ({total_vram_gb:.1f} GB VRAM)")
print(f"Selected Model: {model_name}")

# Automatic Hyperparameter Configuration
if MODEL_SCALE == "3b":
    max_seq_length = 3072
    batch_size = 2
    grad_accum = 4
    learning_rate = 2e-4
    lora_r = 32
    load_4bit = True
elif MODEL_SCALE == "7b":
    max_seq_length = 2048 if total_vram_gb < 20 else 3072
    batch_size = 1 if total_vram_gb < 20 else 2
    grad_accum = 8 if total_vram_gb < 20 else 4
    learning_rate = 1.5e-4
    lora_r = 32
    load_4bit = True
elif MODEL_SCALE in ["14b", "27b", "32b"]:
    if total_vram_gb < 20.0:
        raise RuntimeError(f"{MODEL_SCALE.upper()} requires at least 24GB VRAM (L4/A100). Attached GPU has {total_vram_gb:.1f} GB.")
    max_seq_length = 2048 if total_vram_gb < 45 else 4096
    batch_size = 1
    grad_accum = 16
    learning_rate = 1e-4
    lora_r = 16
    load_4bit = True if total_vram_gb < 75.0 else False  # Can run full 16-bit on A100-80GB!

print(f"-> Configured: 4-bit={load_4bit}, batch_size={batch_size}, grad_accum={grad_accum}, max_seq={max_seq_length}")

# Load Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_4bit,
    dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_r * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model and LoRA adapters successfully initialized!")

In [ ]:
# 5. Load & Format Quad-GPU SFT Dataset
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

template_name = "gemma-2" if "gemma" in model_name.lower() else "qwen-2.5"
tokenizer = get_chat_template(tokenizer, chat_template=template_name)

raw_dataset = load_dataset("json", data_files="./data/sft_multi_gpu.jsonl", split="train")
print(f"Loaded {len(raw_dataset)} verified physical training samples across 4 GPU architectures!")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = raw_dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# 6. Train Stage 1: Supervised Fine-Tuning (SFT)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_ratio=0.08,
        num_train_epochs=3,
        learning_rate=learning_rate,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=f"./checkpoints/gpuperf_{MODEL_SCALE}_sft",
        report_to="none"
    ),
)

print(f"Starting SFT Training for {MODEL_SCALE.upper()}...")
trainer.train()
model.save_pretrained(f"./checkpoints/gpuperf_{MODEL_SCALE}_sft")
tokenizer.save_pretrained(f"./checkpoints/gpuperf_{MODEL_SCALE}_sft")
print(f"✅ SFT Completed and Saved for {MODEL_SCALE.upper()}!")

In [ ]:
# 7. Train Stage 2: Direct Preference Optimization (DPO)
from trl import DPOTrainer, DPOConfig
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

dpo_raw = load_dataset("json", data_files="./data/dpo_multi_gpu.jsonl", split="train")
print(f"Loaded {len(dpo_raw)} empirical DPO preference pairs!")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    tokenizer=tokenizer,
    beta=0.1,
    train_dataset=dpo_raw,
    max_length=max_seq_length,
    args=DPOConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16 if MODEL_SCALE in ["14b", "27b", "32b"] else 8,
        warmup_ratio=0.1,
        num_train_epochs=1,
        learning_rate=5e-6,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        seed=42,
        output_dir=f"./checkpoints/gpuperf_{MODEL_SCALE}_dpo",
        report_to="none"
    ),
)

print(f"Starting DPO Alignment for {MODEL_SCALE.upper()}...")
dpo_trainer.train()

final_dir = f"./checkpoints/gpuperf_{MODEL_SCALE}_dpo_final"
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"🏆 DPO Alignment Finished! Saved final model to {final_dir}")

In [ ]:
# 8. Run 3-Tier Closed-Loop Correctness Gate on Test Kernel
from gpuperf.profiler.correctness_gate import CorrectnessGate

# Select a test prompt from the dataset
test_sample = raw_dataset[0]
inputs = tokenizer.apply_chat_template(test_sample["messages"][:2], tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

FastLanguageModel.for_inference(model)
outputs = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.1)
gen_response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

code_str, launch_cfg = CorrectnessGate.extract_triton_code(gen_response)
print("Extracted Launch Config:", launch_cfg)

gate_results = CorrectnessGate.verify_gemm_kernel(code_str, launch_cfg, baseline_latency_us=13684.9)
print("=== 3-TIER GATE VERIFICATION RESULTS ===")
for k, v in gate_results.items():
    print(f"  * {k}: {v}")